In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score

def redes(file_names,file_names_1000):
    df_list = [pd.read_excel(arquivo) for arquivo in file_names_1000]
    z_preds = [df['Z_pred'].values.reshape(-1, 1) for df in df_list]
    Z_pred_total = np.hstack(z_preds)
    Z_pred_sum = np.sum(Z_pred_total, axis=1).reshape(-1, 1)
    df = pd.read_excel("1000_model/1000_model_8_8_17.xlsx")
    Z = df['Z'].values.reshape(-1, 1)
    mse_sup = np.mean((Z - Z_pred_sum/10) ** 2)

    df_list1 = [pd.read_excel(arquivo) for arquivo in file_names]
    z_preds1 = [df['Z_pred'].values.reshape(-1, 1) for df in df_list1]
    Z_pred_total1 = np.hstack(z_preds1)
    Z_pred_sum1 = np.sum(Z_pred_total1, axis=1).reshape(-1, 1)
    df1 = pd.read_excel("25_model/25_model_8_8_17.xlsx")
    Z1 = df1['Z'].values.reshape(-1, 1)

    mse_sup1 = np.mean((Z1 - Z_pred_sum1/10) ** 2)
    r2_sup_1000 = r2_score(Z, Z_pred_sum/10)
    r2_sup_25 = r2_score(Z1, Z_pred_sum1/10)
    
    print(f"r2_25: {r2_sup_25}")
    print(f"r2_1000: {r2_sup_1000}")
    print(f"mse_1000: {mse_sup}")
    print(f"mse_25: {mse_sup1}")
    return mse_sup

file_names_1000 =  ["1000_model/1000_model_13_6_12.xlsx", "1000_model/1000_model_13_6_2.xlsx",
              "1000_model/1000_model_13_6_20.xlsx", "1000_model/1000_model_13_8_7.xlsx", 
              "1000_model/1000_model_17_5_5.xlsx", "1000_model/1000_model_28_6_0.xlsx", 
              "1000_model/1000_model_32_7_1.xlsx","1000_model/1000_model_33_9_0.xlsx",
              "1000_model/1000_model_33_9_1.xlsx", "1000_model/1000_model_8_8_17.xlsx"]

file_names = [
    "25_model/25_model_13_6_12.xlsx",
    "25_model/25_model_13_6_2.xlsx",
    "25_model/25_model_13_6_20.xlsx",
    "25_model/25_model_13_8_7.xlsx",
    "25_model/25_model_17_5_5.xlsx",
    "25_model/25_model_28_6_0.xlsx",
    "25_model/25_model_32_7_1.xlsx",
    "25_model/25_model_33_9_0.xlsx",
    "25_model/25_model_33_9_1.xlsx",
    "25_model/25_model_8_8_17.xlsx"
]

result= redes(file_names, file_names_1000)




r2_25: 0.9994106081679835
r2_1000: 0.9626550257093741
mse_1000: 0.037630208765278714
mse_25: 0.0005938935052071422


In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt


lambda_reg = 5e-6        
lr = 3e-5                 
n_epocas = 5000000


def redes(file_names):
    df_list = [pd.read_excel(arquivo) for arquivo in file_names]
    z_preds = np.hstack([df['Z_pred'].values.reshape(-1,1) for df in df_list])
    dfZ = pd.read_excel("25_model/25_model_8_8_17.xlsx")
    Z = dfZ['Z'].values.reshape(-1, 1)
    N = Z.shape[0]
    return Z, z_preds, N

def pesos(Z, z_preds, N, patience=500, min_delta=10e-3):  
    
    best_erro = np.inf
    best_w = None
    patience_counter = 0
    a = np.random.uniform(0, 1, z_preds.shape[1])

    for epoch in range(1, n_epocas+1):
        w = np.exp(a) / np.sum(np.exp(a)) # softmax
        yhat = np.dot(z_preds, w)
        residuo = Z.flatten() - yhat
        mse = np.mean(residuo**2)
        mse_pond = mse + lambda_reg * np.sum(a**2)
        # gradiente dmse/dw
        gmse = (-2.0 / N) * z_preds.T.dot(residuo)
        # gradiente completo
        s = np.dot(gmse, w)
        grad_a = w * (gmse - s) + 2.0 * lambda_reg * a
        # atualização
        a = a - lr * grad_a

        # ---- EARLY STOPPING ----
        if mse_pond < best_erro - min_delta:
            best_erro = mse_pond
            best_w = w.copy()
            patience_counter = 0  # reset
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"\nEarly stopping ativado na época {epoch} — perda não melhorou por {patience} épocas.")
            break
        # -------------------------

        if epoch % 100 == 0 or epoch == 1:
            r2 = 1 - np.sum((Z - yhat.reshape(-1,1))**2) / np.sum((Z - Z.mean())**2)
            print(f"epoch {epoch:4d} loss_25={mse_pond:.6f} mse_25={mse:.6f} R2_25={r2:.4f} weights={w}")

    return best_erro, best_w

def mse(file_names, file_names_1000, best_w):
    df_list = [pd.read_excel(arquivo) for arquivo in file_names_1000]
    z_preds = [df['Z_pred'].values.reshape(-1, 1) for df in df_list]
    Z_pred_total = np.hstack(z_preds)
    yhat_final = Z_pred_total * best_w      
    Z_pred_sum = np.sum(yhat_final, axis=1).reshape(-1, 1)
    df1 = pd.read_excel("1000_model/1000_model_8_8_17.xlsx")
    Z1 = df1['Z'].values.reshape(-1, 1)
    df_list2 = [pd.read_excel(arquivo) for arquivo in file_names]
    z_preds2 = [df['Z_pred'].values.reshape(-1, 1) for df in df_list2]
    Z_pred_total2 = np.hstack(z_preds2)
    yhat_final2 = Z_pred_total2 * best_w      
    Z_pred_sum2 = np.sum(yhat_final2, axis=1).reshape(-1, 1)
    df2 = pd.read_excel("25_model/25_model_8_8_17.xlsx")
    Z2 = df2['Z'].values.reshape(-1, 1)

    mse_sup_1000 = np.mean((Z1 - Z_pred_sum) ** 2)
    mse_25 = np.mean((Z2 - Z_pred_sum2) ** 2)
    r2_sup_1000 = r2_score(Z1, Z_pred_sum)
    r2_sup_25 = r2_score(Z2, Z_pred_sum2)

    print(f"MSE conjunto = {mse_sup_1000}")
    print(f"MSE 25 dados = {mse_25}")
    print(f"R² conjunto = {r2_sup_1000}")
    print(f"R² 25 dados = {r2_sup_25}")

    return mse_sup_1000, r2_sup_1000, mse_25

file_names_1000 =  ["1000_model/1000_model_13_6_12.xlsx", "1000_model/1000_model_13_6_2.xlsx",
              "1000_model/1000_model_13_6_20.xlsx", "1000_model/1000_model_13_8_7.xlsx", 
              "1000_model/1000_model_17_5_5.xlsx", "1000_model/1000_model_28_6_0.xlsx", 
              "1000_model/1000_model_32_7_1.xlsx","1000_model/1000_model_33_9_0.xlsx",
              "1000_model/1000_model_33_9_1.xlsx", "1000_model/1000_model_8_8_17.xlsx"]

file_names = [
    "25_model/25_model_13_6_12.xlsx",
    "25_model/25_model_13_6_2.xlsx",
    "25_model/25_model_13_6_20.xlsx",
    "25_model/25_model_13_8_7.xlsx",
    "25_model/25_model_17_5_5.xlsx",
    "25_model/25_model_28_6_0.xlsx",
    "25_model/25_model_32_7_1.xlsx",
    "25_model/25_model_33_9_0.xlsx",
    "25_model/25_model_33_9_1.xlsx",
    "25_model/25_model_8_8_17.xlsx"
]

Z, z_preds, N= redes(file_names)
best_erro, best_w = pesos(Z, z_preds, N)

print("best erro =", best_erro)
print("best pesos =", best_w)

mse_1000 =mse(file_names, file_names_1000, best_w)



epoch    1 loss_25=0.000499 mse_25=0.000480 R2_25=0.9995 weights=[0.09789908 0.06947056 0.06994396 0.14201044 0.11356345 0.08088147
 0.10001402 0.0926148  0.08919262 0.14440962]
epoch  100 loss_25=0.000499 mse_25=0.000480 R2_25=0.9995 weights=[0.09789905 0.06947054 0.06994394 0.14201045 0.11356348 0.08088148
 0.10001401 0.09261478 0.08919261 0.14440965]
epoch  200 loss_25=0.000499 mse_25=0.000480 R2_25=0.9995 weights=[0.09789902 0.06947053 0.06994393 0.14201047 0.11356352 0.0808815
 0.100014   0.09261476 0.08919259 0.14440968]
epoch  300 loss_25=0.000499 mse_25=0.000480 R2_25=0.9995 weights=[0.097899   0.06947052 0.06994392 0.14201048 0.11356355 0.08088151
 0.10001399 0.09261474 0.08919258 0.14440971]
epoch  400 loss_25=0.000499 mse_25=0.000480 R2_25=0.9995 weights=[0.09789897 0.06947051 0.0699439  0.14201049 0.11356358 0.08088153
 0.10001398 0.09261473 0.08919257 0.14440974]
epoch  500 loss_25=0.000499 mse_25=0.000480 R2_25=0.9995 weights=[0.09789895 0.06947049 0.06994389 0.1420105  0

In [3]:
lambda_reg = 5e-6        
lr = 1e-2                 
n_epocas = 5000000

best_erro1, best_w_1 = pesos(Z, z_preds, N)

print("best erro =", best_erro1)
print("best pesos =", best_w_1)
mse_1000 =mse(file_names, file_names_1000, best_w_1)

epoch    1 loss_25=0.000507 mse_25=0.000494 R2_25=0.9995 weights=[0.0826122  0.08887079 0.07244271 0.0716919  0.13263229 0.0947711
 0.07288865 0.08697142 0.13856858 0.15855037]
epoch  100 loss_25=0.000507 mse_25=0.000493 R2_25=0.9995 weights=[0.0826064  0.08886407 0.07243827 0.07169432 0.1326492  0.09477687
 0.07288706 0.08696439 0.13855408 0.15856535]
epoch  200 loss_25=0.000507 mse_25=0.000493 R2_25=0.9995 weights=[0.08260054 0.08885728 0.07243379 0.07169677 0.13266627 0.0947827
 0.07288545 0.08695729 0.13853943 0.15858049]
epoch  300 loss_25=0.000507 mse_25=0.000493 R2_25=0.9995 weights=[0.08259468 0.08885049 0.0724293  0.07169921 0.13268335 0.09478852
 0.07288384 0.08695019 0.13852479 0.15859562]
epoch  400 loss_25=0.000507 mse_25=0.000493 R2_25=0.9995 weights=[0.08258882 0.0888437  0.07242482 0.07170166 0.13270043 0.09479435
 0.07288223 0.0869431  0.13851015 0.15861074]
epoch  500 loss_25=0.000507 mse_25=0.000493 R2_25=0.9995 weights=[0.08258296 0.08883692 0.07242033 0.0717041  0.

In [4]:
lambda_reg = 5e-6        
lr = 1e-2                 
n_epocas = 5000000

best_erro1, best_w_1 = pesos(Z, z_preds, N)

print("best erro =", best_erro1)
print("best pesos =", best_w_1)
mse_1000 =mse(file_names, file_names_1000, best_w_1)

epoch    1 loss_25=0.000629 mse_25=0.000609 R2_25=0.9994 weights=[0.09393768 0.11510812 0.08080331 0.09361514 0.0693668  0.13735219
 0.12566781 0.10951001 0.08580639 0.08883256]
epoch  100 loss_25=0.000629 mse_25=0.000609 R2_25=0.9994 weights=[0.09393051 0.11509711 0.08079814 0.09362249 0.06937355 0.13736754
 0.12566581 0.10950193 0.08580299 0.08883992]
epoch  200 loss_25=0.000629 mse_25=0.000609 R2_25=0.9994 weights=[0.09392328 0.11508599 0.08079292 0.09362992 0.06938037 0.13738304
 0.1256638  0.10949377 0.08579956 0.08884736]
epoch  300 loss_25=0.000629 mse_25=0.000609 R2_25=0.9994 weights=[0.09391604 0.11507487 0.0807877  0.09363735 0.0693872  0.13739854
 0.12566178 0.10948561 0.08579613 0.08885479]
epoch  400 loss_25=0.000629 mse_25=0.000609 R2_25=0.9994 weights=[0.09390881 0.11506375 0.08078248 0.09364477 0.06939402 0.13741404
 0.12565976 0.10947745 0.08579269 0.08886222]
epoch  500 loss_25=0.000629 mse_25=0.000609 R2_25=0.9994 weights=[0.09390158 0.11505263 0.08077726 0.0936522  